# Getting started with Parse endpoint

Cohere's Parse is a vision model that converts unstructured enterprise documents into structured output for AI knowledge applications such as search & retrieval and RAG pipelines.

It extracts text and reading order, tables, lists, forms and key-value pairs, and images with captions. With `output_format="blocks"`, each page returns an ordered list of content blocks — `text`, `image`, and `table` — including bounding boxes and image categories (e.g. `logo`, `signature`, `flowchart`).

Docs: https://docs.cohere.com/v2/docs/parse

> Note: the API accepts images as base64 data URIs. Sample images are re-encoded to plain JPEG with Pillow before sending.

In [ ]:
import os, io, json, base64
from dotenv import load_dotenv
from PIL import Image
import cohere
from IPython.display import display, Image as IPyImage
from pygments import highlight
from pygments.lexers import JsonLexer
from pygments.formatters import TerminalFormatter

load_dotenv(override=True)

co = cohere.ClientV2(
    base_url=os.environ["PARSE_BASE_URL"],
    api_key=os.environ["PARSE_API_KEY"],
    timeout=300,
)
MODEL = os.environ["PARSE_MODEL"]

def to_data_uri(path):
    """Re-encode an image as a JPEG base64 data URI (also converts JPEG 2000)."""
    im = Image.open(path).convert("RGB")
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def prune(obj):
    """Recursively drop null fields for a cleaner JSON view."""
    if isinstance(obj, dict):
        return {k: prune(v) for k, v in obj.items() if v is not None}
    if isinstance(obj, list):
        return [prune(v) for v in obj]
    return obj

def parse_file(path):
    """Display the input image, call Parse with blocks output, show indented JSON."""
    display(IPyImage(filename=str(path), width=480))
    response = co.parse(
        model=MODEL,
        document={"type": "image_url", "image_url": to_data_uri(path)},
        output_format="blocks",
    )
    data = prune(response.model_dump())
    for page in data.get("pages", []):
        for block in page.get("blocks", []):
            for value in block.values():
                if isinstance(value, dict) and "html" in value:
                    value["html"] = value["html"][:120] + " ..."
    # 16-color ANSI adapts to light/dark themes; pygments "friendly" hardcodes light-theme colors
    print(highlight(json.dumps(data, indent=2, default=str), JsonLexer(), TerminalFormatter()))

print('*** Finish running cell, no output for this cell')

## Input 1: scanned CD document (with signature)

In [ ]:
print("*** This is the first parse endpoint call. If you get errors here, check your .env file for correctness.\n")
parse_file("sample_inputs/cd_scanned_with_signature.jpg")

## Input 2: healthcare cost report

In [ ]:
parse_file("sample_inputs/healthcare_cost_report.jpg")

## Important
Once you complete a notebook select **Restart** at the top of VS Code to restart the kernel before moving to the next notebook.

## Optional: calling the /parse endpoint directly over HTTP

Under the hood, the `cohere` SDK simply POSTs to the deployment's `/v2/parse` route with your API key. Microsoft's Foundry inference SDK (`azure-ai-inference`) currently only exposes chat-completions and embeddings clients — Parse is invoked via the Cohere SDK or plain HTTP.

This cell shows the raw HTTP version with `requests` — useful for debugging or non-Python clients. (Serverless endpoints on `*.inference.ai.azure.com` use an `api-key: <key>` header instead of `Authorization: Bearer`.)

In [ ]:
import os, io, json, base64, requests
from PIL import Image
from IPython.display import display, Image as IPyImage
from pygments import highlight
from pygments.lexers import JsonLexer
from pygments.formatters import TerminalFormatter

def prune(obj):
    """Recursively drop null fields for a cleaner JSON view."""
    if isinstance(obj, dict):
        return {k: prune(v) for k, v in obj.items() if v is not None}
    if isinstance(obj, list):
        return [prune(v) for v in obj]
    return obj

def parse_file_http(path):
    """Parse an image via a raw POST to the /parse route — no SDK needed."""
    url = os.environ["PARSE_BASE_URL"].rstrip("/") + "/v2/parse"
    im = Image.open(path).convert("RGB")
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=90)
    image_b64 = base64.b64encode(buf.getvalue()).decode()
    display(IPyImage(filename=str(path), width=480))
    response = requests.post(
        url,
        headers={"api-key": os.environ["PARSE_API_KEY"]},
        json={
            "model": os.environ["PARSE_MODEL"],
            "document": {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
            "output_format": "blocks",
        },
        timeout=300,
    )
    response.raise_for_status()
    data = prune(response.json())
    for page in data.get("pages", []):
        for block in page.get("blocks", []):
            for value in block.values():
                if isinstance(value, dict) and "html" in value:
                    value["html"] = value["html"][:120] + " ..."
    print(highlight(json.dumps(data, indent=2), JsonLexer(), TerminalFormatter()))

parse_file_http("sample_inputs/cd_scanned_with_signature.jpg")

In [ ]:
parse_file_http("sample_inputs/healthcare_cost_report.jpg")